# PersonaPlex + IMTalker — RunPod Live Streaming Notebook

**Pipeline (from source code):**
```
Browser mic (48 kHz PCM, binary WS)
  └─> MoshiOnlyEngineWithHidden  (liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py)
        Mimi encoder  ──> PersonaPlex 7B LM (bnb-4bit)  ──> Mimi decoder
        Layer[-2] hidden states  [B=1, 1, 4096]  @ 12.5 Hz
  └─> StudioNativeLiveAdapter  (helium_w2v_frontend_adapter.py + wav2vec2.py)
        HeliumToWav2VecFrontendAdapter (6-layer Transformer)  ->  [T×768]
        Frozen Wav2Vec2.encode_from_projected_frontend()      ->  [T×768]
  └─> FMGenerator.sample()  (generator/FM.py)
        audio_projection(768→32) + FlowMatchingTransformer + ODE  ->  motion latents [T×32]
  └─> IMTRenderer  (renderer/models.py)
        dense_feature_encoder(ref_img)  ->  identity features
        adapt(motion, g_r)  ->  latent_token_decoder  ->  motion maps
        decode(motion_maps, ref_motion_maps, ref_features)  ->  512×512 RGB @ 25 fps
  └─> ws_av_binary_codec.pack_av_frame()  ->  binary WebSocket -> browser
```

**Server module (updated):** this notebook now launches `IMTalker/personaplex_imtalker_streaming_server.py`, a FlashTalk-v3-architecture rewrite of the pipeline above. It keeps the exact same models/checkpoints/wire protocol shown in the diagram, but splits PersonaPlex generation and IMTalker rendering into two independent threads (`PersonaPlexThread` / `IMTalkerThread`) joined by a queue, instead of running both serially in one thread as the legacy `liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py` did. That legacy file is untouched and still used by the offline tools (`offline_personaplex_imtalker_infer.py`, `render_saved_live_helium.py`).

**Checkpoints required:**
| Checkpoint | Path |
|---|---|
| IMTalker generator | `/workspace/IMTalker/checkpoints/generator.ckpt` |
| IMTalker renderer | `/workspace/IMTalker/checkpoints/renderer.ckpt` |
| PersonaPlex bnb-4bit | `/workspace/personaplex_bnb4/model_bnb_4bit.pt` |
| Helium→Wav2Vec2 adapter | `/workspace/exps/personaplex_frontend_adapter/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt` |
| Wav2Vec2-base-960h | `/workspace/IMTalker/checkpoints/wav2vec2-base-960h/` |
| Mimi tokenizer | from HF `nvidia/personaplex-7b-v1` |
| Voice prompt | `NATM0.pt` from HF `nvidia/personaplex-7b-v1` `voices.tgz` |
| Reference image | `/workspace/IMTalker/assets/2_vid_robert.png` |

**Source files used:**
- `IMTalker/personaplex_imtalker_streaming_server.py` -- **current** main entrypoint (FlashTalk-v3 architecture; launched by this notebook)
- `IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py` — legacy entrypoint (unmodified; still used by offline tools)
- `IMTalker/liveTry.py` — `MoshiOnlyEngine` base class
- `IMTalker/generator/FM.py` — `FMGenerator`
- `IMTalker/generator/helium_w2v_frontend_adapter.py` — `HeliumToWav2VecFrontendAdapter`
- `IMTalker/generator/wav2vec2.py` — `Wav2VecModel`
- `IMTalker/renderer/models.py` — `IMTRenderer`
- `IMTalker/ws_av_binary_codec.py` — binary frame packing
- `IMTalker/run_personaplex_imtalker_source5_8998.sh` — canonical run arguments
- `personaplex/moshi/moshi/models/loaders.py` — model loaders
- `personaplex/moshi/moshi/models/lm.py` — `LMModel` / `LMGen`

In [1]:
!git clone https://github.com/MoshiHead/IMTalker-PersonaPlex-Streaming-v1.git

Cloning into 'IMTalker-PersonaPlex-Streaming-v1'...
remote: Enumerating objects: 212, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 212 (delta 10), reused 209 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (212/212), 15.69 MiB | 41.09 MiB/s, done.
Resolving deltas: 100% (10/10), done.


## Cell 1 — System & CUDA Validation

In [2]:
import subprocess, sys, os, shutil, json, time, pathlib

def _run(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    return r.stdout.strip(), r.returncode

print('=== Python ===')
print(sys.version)

print('\n=== CUDA / PyTorch ===')
import torch
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count: {n}')
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU {i}: {props.name}  VRAM={vram_gb:.1f} GB')
        if vram_gb < 16:
            print(f'  WARNING: GPU {i} has {vram_gb:.1f} GB — recommend >=24 GB for bnb-4bit PersonaPlex')
else:
    raise EnvironmentError('No CUDA GPU detected. This pipeline requires CUDA.')

print('\n=== FFmpeg ===')
ffmpeg_path = shutil.which('ffmpeg')
if ffmpeg_path:
    out, _ = _run('ffmpeg -version')
    print(out.splitlines()[0])
else:
    print('ffmpeg not found — installing...')
    os.system('apt-get install -y ffmpeg -qq')

print('\n=== torchaudio ===')
import torchaudio
print(f'torchaudio: {torchaudio.__version__}')

print('\nSystem validation passed.')

=== Python ===
3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]

=== CUDA / PyTorch ===
torch: 2.8.0+cu128
CUDA available: True
GPU count: 1
  GPU 0: NVIDIA A100-SXM4-80GB  VRAM=79.3 GB

=== FFmpeg ===
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers

=== torchaudio ===
torchaudio: 2.8.0+cu128

System validation passed.


## Cell 2 — Repository Path Validation

In [3]:
# Expected workspace layout (from run_personaplex_imtalker_source5_8998.sh):
#   /workspace/IMTalker                               <- IMTalker repo root
#   /workspace/personaplex_bnb4                       <- PersonaPlex moshi package + weights
#   /workspace/personaplex_bnb4/moshi                <- moshi Python package (in PYTHONPATH)
#   /workspace/exps/personaplex_frontend_adapter/...  <- adapter checkpoint

IMTALKER_ROOT   = '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker'
PERSONAPLEX_ROOT = '/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex'
MOSHI_PKG       = '/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi'

missing = []
for p in [IMTALKER_ROOT, PERSONAPLEX_ROOT, MOSHI_PKG]:
    exists = pathlib.Path(p).exists()
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {p}')
    if not exists:
        missing.append(p)

if missing:
    raise FileNotFoundError(
        f'Missing required directories: {missing}\n'
        'Clone/copy the repos to /workspace before proceeding.\n'
        '  git clone <imtalker-repo> /workspace/IMTalker\n'
        '  git clone <personaplex-bnb4-repo> /workspace/personaplex_bnb4'
    )

# Check that key source files exist
MAIN_SCRIPT = f'{IMTALKER_ROOT}/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py'
required_sources = [
    MAIN_SCRIPT,
    f'{IMTALKER_ROOT}/liveTry.py',
    f'{IMTALKER_ROOT}/ws_av_binary_codec.py',
    f'{IMTALKER_ROOT}/generator/FM.py',
    f'{IMTALKER_ROOT}/generator/helium_w2v_frontend_adapter.py',
    f'{IMTALKER_ROOT}/generator/wav2vec2.py',
    f'{IMTALKER_ROOT}/generator/options/base_options.py',
    f'{IMTALKER_ROOT}/renderer/models.py',
    f'{IMTALKER_ROOT}/static/index_v3_binary_fullscreen.html',
    f'{MOSHI_PKG}/moshi/models/loaders.py',
    f'{MOSHI_PKG}/moshi/models/lm.py',
]
print()
src_missing = []
for f in required_sources:
    exists = pathlib.Path(f).is_file()
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {f}')
    if not exists:
        src_missing.append(f)

if src_missing:
    raise FileNotFoundError(f'Missing source files: {src_missing}')

print('\nAll repository paths validated.')

[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi

[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/liveTry.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/ws_av_binary_codec.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/generator/FM.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/generator/helium_w2v_frontend_adapter.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/generator/wav2vec2.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/generator/options/base_options.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/renderer/models.py
[OK] /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/static/index_v3_binary_fullscreen.html
[OK] /workspace/IMT

## Cell 3 — Dependency Installation

Packages derived from `IMTalker/requirement.txt` plus runtime imports traced in the source files.

In [4]:
# Core packages from IMTalker/requirement.txt
# Plus runtime dependencies traced from source:
#   liveTry.py         -> sentencepiece, huggingface_hub
#   liveTryHelium...py -> fastapi, uvicorn, transformers (Wav2Vec2FeatureExtractor)
#   generator/FM.py    -> torchdiffeq
#   loaders.py         -> safetensors, bitsandbytes (quantize_4bit path)
#   _apply_system_prompts -> sentencepiece
#
# Python 3.12 + tokenizers:
#   requirement.txt pins transformers==4.30.2 (tokenizers<0.14), but all tokenizers
#   releases <0.14 predate Python 3.12 and have no cp312 wheel — they require Rust.
#   tokenizers==0.13.4 does not exist on PyPI (jumps 0.13.3 → 0.14.0).
#   Fix: transformers>=4.36.0 allows tokenizers>=0.14 (first cp312 prebuilt series).
#
# numpy / ABI note:
#   RunPod pods with torch 2.8.0+cu128 pre-install numpy 2.x and load it into the
#   kernel at startup.  If we then downgrade numpy to 1.x on disk (via numpy<2.0.0),
#   C extensions just installed by pip (compiled against numpy 1.x, 96-byte dtype)
#   crash against the in-memory numpy 2.x (88-byte dtype):
#     "numpy.dtype size changed: Expected 96 from C header, got 88 from PyObject"
#   numpy.rec errors are from old scipy/pandas that still import the 1.x submodule.
#   Fix: keep numpy 2.x; install numpy-2.x-compatible versions of all packages.
#
# moshi namespace conflict:
#   The standard kyutai moshi package (if installed) and the PersonaPlex fork both
#   expose the `moshi` Python namespace.  If the standard one is present in
#   site-packages, Python may resolve `from moshi.models import loaders` to the
#   standard loaders which lack the `quantize_4bit` parameter → TypeError.
#   Fix: uninstall standard moshi before installing the PersonaPlex editable fork.
#
# moshi editable install:
#   moshi-personaplex 0.1.0 declares torch<2.5 and huggingface-hub<0.25.
#   Fix: install with --no-deps so it cannot downgrade the pod's torch or hub.

import subprocess, sys, os, pathlib

# ── Step 0: upgrade pip ───────────────────────────────────────────────────────
print('Upgrading pip...')
ret0 = os.system(f'{sys.executable} -m pip install -q --upgrade pip')
if ret0 != 0:
    print('WARNING: pip upgrade returned non-zero — continuing anyway.')
else:
    print('pip upgraded.')

# ── Step 1: snapshot pre-install state (via metadata, not module reload) ──────
import importlib.metadata, importlib, torch as _t0
_torch_ver_before = importlib.metadata.version('torch')
_np_ver_before    = importlib.metadata.version('numpy')
print(f'Pre-install: torch={_torch_ver_before}  numpy={_np_ver_before}  CUDA={_t0.cuda.is_available()}')
del _t0

# ── Step 2: batch-install all packages ───────────────────────────────────────
packages = [
    # numpy: no upper cap — pod pre-installs numpy 2.x; don't downgrade it.
    # (requirement.txt said <2.0.0 but that predates the numpy 2.x ABI change)
    'numpy>=1.24.0',
    # pandas 2.x is numpy 2.x compatible; pandas 1.x uses numpy.rec which was
    # removed from the default namespace in numpy 2.x.
    'pandas>=2.0.0,<2.2.0',
    'matplotlib<3.9.0',
    # opencv 4.9+ added numpy 2.x ABI support.
    'opencv-python>=4.9.0',
    'av==12.0.0',
    'flow-vis',
    'albucore>=0.0.16',
    'face_alignment==1.4.1',
    # scipy 1.11+ dropped numpy.rec usage and ships numpy 2.x compatible wheels.
    'scipy>=1.11.0',
    'transformers>=4.36.0,<5.0.0',  # 4.36+ allows tokenizers>=0.14 (cp312 prebuilt)
    'pytorch-lightning==2.2.1',
    'torchdiffeq==0.2.5',           # generator/FM.py: from torchdiffeq import odeint
    'timm==1.0.9',
    'pyyaml',
    'tqdm',
    'librosa',
    'tensorboard',
    'fastapi==0.115.6',
    'pydantic==2.10.4',
    # Runtime deps not in requirement.txt
    'uvicorn[standard]',            # liveTry.py main(): uvicorn.run(...)
    'websockets',
    'httpx',
    'sentencepiece',                # liveTry.py: SentencePieceProcessor(tokenizer)
    'huggingface-hub>=0.20.0',      # liveTry.py: hf_hub_download
    'safetensors>=0.4.0',           # loaders.py: from safetensors.torch import load_model
    'bitsandbytes>=0.43.0',         # loaders.get_moshi_lm(quantize_4bit=True)
    'accelerate>=0.26.0',           # required by bitsandbytes 4-bit quantization
    'pillow',
]

pip_cmd = (
    f'{sys.executable} -m pip install -q '
    + ' '.join(f'"{p}"' for p in packages)
)
print('\nInstalling packages (this may take a few minutes)...')
ret = os.system(pip_cmd)
if ret != 0:
    print('WARNING: pip returned non-zero. Check output above for errors.')
else:
    print('Package installation complete.')

# ── Step 3: verify key versions ───────────────────────────────────────────────
importlib.invalidate_caches()
tok_ver = importlib.metadata.version('tokenizers')
tfm_ver = importlib.metadata.version('transformers')
np_ver  = importlib.metadata.version('numpy')
print(f'\ntokenizers:   {tok_ver}')
print(f'transformers: {tfm_ver}')
print(f'numpy:        {np_ver}')

from packaging.version import Version
if Version(tok_ver) < Version('0.14'):
    raise RuntimeError(
        f'tokenizers=={tok_ver} has no cp312 prebuilt wheel and requires Rust.\n'
        'Force install: pip install "transformers>=4.36.0,<5.0.0" "tokenizers>=0.14"'
    )
print('tokenizers version OK (>= 0.14).')

if Version(np_ver) < Version('2.0'):
    print(f'numpy {np_ver} is < 2.0 — C-extension ABI mismatch may occur if the '
          'kernel loaded numpy 2.x at startup. Restart the kernel and re-run this cell '
          'to let all packages load under numpy 1.x consistently.')
else:
    print(f'numpy {np_ver} is >= 2.0 — all installed packages are numpy-2.x-compatible.')

# ── Step 4: remove standard moshi to prevent namespace conflict ───────────────
# The standard kyutai `moshi` package (if installed) exposes the same `moshi`
# Python namespace as the PersonaPlex fork.  If it appears in site-packages,
# `from moshi.models import loaders` in liveTry.py resolves to the standard
# loaders which have no `quantize_4bit` → TypeError at model load time.
# Uninstalling it here ensures the editable PersonaPlex fork is the only provider.
print('\nRemoving standard moshi (if installed) to prevent namespace conflict...')
os.system(f'{sys.executable} -m pip uninstall -y moshi > /dev/null 2>&1')
# (exits non-zero if not installed — that is expected and fine)
print('Standard moshi removed (or was not present).')

# ── Step 5: install the personaplex moshi package — NO DEPS ──────────────────
# --no-deps prevents moshi-personaplex 0.1.0 from downgrading torch/huggingface-hub.
moshi_setup     = pathlib.Path(MOSHI_PKG) / 'setup.cfg'
moshi_pyproject = pathlib.Path(MOSHI_PKG) / 'pyproject.toml'
if moshi_setup.exists() or moshi_pyproject.exists():
    print(f'Installing PersonaPlex moshi (--no-deps) from {MOSHI_PKG} ...')
    ret2 = os.system(f'{sys.executable} -m pip install -q --no-deps -e "{MOSHI_PKG}"')
    if ret2 != 0:
        print('WARNING: moshi editable install returned non-zero.')
    else:
        print('PersonaPlex moshi installed (no-deps).')
else:
    if MOSHI_PKG not in sys.path:
        sys.path.insert(0, MOSHI_PKG)
    print(f'Added {MOSHI_PKG} to sys.path (no setup.cfg/pyproject.toml found)')

# ── Step 6: verify torch was NOT downgraded ───────────────────────────────────
importlib.invalidate_caches()
_torch_ver_after = importlib.metadata.version('torch')
print(f'\nPost-install torch metadata: {_torch_ver_after}')
if _torch_ver_after != _torch_ver_before:
    print(f'WARNING: torch changed {_torch_ver_before} -> {_torch_ver_after}')
    print('Attempting to restore...')
    os.system(f'{sys.executable} -m pip install -q --no-deps "torch=={_torch_ver_before}"')
    print('Restore attempted. Restart the kernel if CUDA is still unavailable.')

import torch
if not torch.cuda.is_available():
    raise EnvironmentError(
        f'CUDA lost after installation (torch={torch.__version__}).\n'
        'Restart the kernel and re-run this cell.'
    )
print(f'torch in process: {torch.__version__}  CUDA=True  — OK')
print('\nDependency installation complete.')

Upgrading pip...
pip upgraded.
Pre-install: torch=2.8.0+cu128  numpy=2.1.2  CUDA=True

Installing packages (this may take a few minutes)...
Package installation complete.

tokenizers:   0.22.2
transformers: 4.57.6
numpy:        1.26.4
tokenizers version OK (>= 0.14).
numpy 1.26.4 is < 2.0 — C-extension ABI mismatch may occur if the kernel loaded numpy 2.x at startup. Restart the kernel and re-run this cell to let all packages load under numpy 1.x consistently.

Removing standard moshi (if installed) to prevent namespace conflict...
Standard moshi removed (or was not present).
Installing PersonaPlex moshi (--no-deps) from /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi ...
PersonaPlex moshi installed (no-deps).

Post-install torch metadata: 2.8.0+cu128
torch in process: 2.8.0+cu128  CUDA=True  — OK

Dependency installation complete.


```
## Cell 4 — Python Path Configuration

PYTHONPATH from `run_personaplex_imtalker_source5_8998.sh`:
```bash
export PYTHONPATH=/workspace/IMTalker:/workspace/personaplex_bnb4/moshi

In [4]:
# Inject the same PYTHONPATH that the run script exports.
# Order matters: IMTalker must come first so its local ws_av_binary_codec.py
# and generator/ packages are found before any site-packages.

import sys, os, importlib, importlib.util as _ilu, pathlib

# ── Paths — update these if your repo is in a different location ─────────────
PATHS_TO_ADD = [
    '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker',          # main source root (liveTry.py, ws_av_binary_codec, generator/, renderer/)
    '/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi',  # moshi Python package
]
# If you cloned the combined repo:
#   PATHS_TO_ADD = [
#       '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker',
#       '/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi',
#   ]

for p in PATHS_TO_ADD:
    if p not in sys.path:
        sys.path.insert(0, p)
    print(f'sys.path += {p}')

# ── Flush any cached standard moshi from sys.modules ─────────────────────────
# The standard kyutai `moshi` package may have been imported as a side-effect
# of the batch install in Cell 3, or it may persist from a previous kernel state.
# Both the standard moshi and the PersonaPlex fork expose the `moshi` namespace.
# If the standard version is cached in sys.modules, `from moshi.models import
# loaders` in liveTry.py will get the cached standard loaders — which do NOT
# have the `quantize_4bit` parameter — causing a TypeError when building the app.
# Clearing the cache forces a fresh lookup, which will find the PersonaPlex fork
# first because PATHS_TO_ADD[-1] was just inserted at sys.path[0].
_moshi_cached = [k for k in list(sys.modules.keys())
                 if k == 'moshi' or k.startswith('moshi.')]
for _k in _moshi_cached:
    del sys.modules[_k]
if _moshi_cached:
    print(f'\nCleared {len(_moshi_cached)} cached moshi modules: {_moshi_cached}')
else:
    print('\nNo cached moshi modules in sys.modules (clean state)')

# ── Confirm which moshi will be found on next import ─────────────────────────
_moshi_spec = _ilu.find_spec('moshi')
if _moshi_spec and _moshi_spec.origin:
    _moshi_origin = str(_moshi_spec.origin)
    print(f'moshi resolves to: {_moshi_origin}')
    _expected = PATHS_TO_ADD[-1]  # personaplex moshi path
    if _expected in _moshi_origin:
        print('  [OK] PersonaPlex fork confirmed')
    else:
        print(f'  [WARN] moshi is NOT from the PersonaPlex fork.')
        print(f'  Expected inside: {_expected}')
        print(f'  Got:             {_moshi_origin}')
        print('  This means `get_moshi_lm(quantize_4bit=...)` will raise TypeError.')
        print('  Likely cause: the standard kyutai moshi is installed in site-packages.')
        print('  Fix: in Cell 3, run:')
        print(f'    pip uninstall -y moshi')
        print(f'    pip install --no-deps -e "{PATHS_TO_ADD[-1]}"')
else:
    print('[WARN] moshi not found by importlib.util.find_spec — check PATHS_TO_ADD[-1]')

# ── Environment variables ─────────────────────────────────────────────────────
# Also set in the subprocess environment so the server process inherits it
existing_pp = os.environ.get('PYTHONPATH', '')
new_pp = ':'.join(PATHS_TO_ADD)
if existing_pp:
    new_pp = new_pp + ':' + existing_pp
os.environ['PYTHONPATH'] = new_pp
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f'\nPYTHONPATH={os.environ["PYTHONPATH"]}')
print(f'PYTORCH_CUDA_ALLOC_CONF={os.environ["PYTORCH_CUDA_ALLOC_CONF"]}')
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}')

# ── Import smoke test ─────────────────────────────────────────────────────────
print('\nSmoke-testing imports...')
for mod in ['ws_av_binary_codec', 'generator.FM', 'generator.wav2vec2',
            'generator.helium_w2v_frontend_adapter', 'renderer.models']:
    try:
        importlib.import_module(mod)
        print(f'  [OK] {mod}')
    except Exception as e:
        print(f'  [FAIL] {mod}: {e}')

print('\nPath configuration complete.')

sys.path += /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker
sys.path += /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi

No cached moshi modules in sys.modules (clean state)
moshi resolves to: /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi/moshi/__init__.py
  [OK] PersonaPlex fork confirmed

PYTHONPATH=/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker:/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/moshi
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
CUDA_VISIBLE_DEVICES=0

Smoke-testing imports...
  [OK] ws_av_binary_codec
  [OK] generator.FM
  [OK] generator.wav2vec2
  [OK] generator.helium_w2v_frontend_adapter
  [OK] renderer.models

Path configuration complete.


## Cell 5 — HuggingFace Authentication

In [ ]:
import os

# The run script requires HF_TOKEN to be set:
#   : "${HF_TOKEN:?Set HF_TOKEN in the environment before running this script}"
# Set your token here OR pass it as an environment variable before running this cell.

#hf-token
# aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ
HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    # Uncomment and fill in your token if not already in the environment:
    # HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'
    raise EnvironmentError(
        'HF_TOKEN is not set.\n'
        'Run in your terminal before launching Jupyter:\n'
        '  export HF_TOKEN=hf_XXXX\n'
        'Or set it in this cell: HF_TOKEN = "hf_XXXX"'
    )

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# Authenticate with huggingface-hub
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print(f'HuggingFace authenticated (token starts with hf_...{HF_TOKEN[-4:]})')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace authenticated (token starts with hf_...rTXQ)


## Cell 6 — Checkpoint Validation & HuggingFace Asset Downloads

Validates all checkpoints identified in `run_personaplex_imtalker_source5_8998.sh`.
Downloads Mimi and voice prompts from `nvidia/personaplex-7b-v1` if missing.

In [7]:
%%bash
set -e

cd "${REPO_DIR:-/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker}"

mkdir -p checkpoints

# Disable hf_transfer if not installed
export HF_HUB_ENABLE_HF_TRANSFER=0

python - <<'PY'
import os
from huggingface_hub import snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("Downloading IMTalker checkpoints...")

snapshot_download(
    repo_id="cbsjtu01/IMTalker",
    repo_type="model",
    local_dir="checkpoints",
    resume_download=True,
)

print("Download complete.")
PY

echo
echo "== checkpoints tree =="
ls -lah checkpoints

echo
if [ -d checkpoints/wav2vec2-base-960h ]; then
    echo "== wav2vec2-base-960h =="
    ls -lah checkpoints/wav2vec2-base-960h
else
    echo "(wav2vec2-base-960h folder not found)"
fi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 9 files: 100%|██████████| 9/9 [00:11<00:00,  1.31s/it]


Download complete.

== checkpoints tree ==
total 2.6G
drwxr-xr-x 4 root root  179 Jun  6 04:43 .
drwxr-xr-x 8 root root 4.0K Jun  6 04:43 ..
drwxr-xr-x 3 root root   33 Jun  6 04:43 .cache
-rw-r--r-- 1 root root 1.5K Jun  6 04:43 .gitattributes
-rw-r--r-- 1 root root   24 Jun  6 04:43 README.md
-rw-r--r-- 1 root root  320 Jun  6 04:43 config.yaml
-rw-r--r-- 1 root root 593M Jun  6 04:43 generator.ckpt
-rw-r--r-- 1 root root 2.0G Jun  6 04:43 renderer.ckpt
drwxr-xr-x 2 root root  139 Jun  6 04:43 wav2vec2-base-960h

== wav2vec2-base-960h ==
total 361M
drwxr-xr-x 2 root root  139 Jun  6 04:43 .
drwxr-xr-x 4 root root  179 Jun  6 04:43 ..
-rw-r--r-- 1 root root 1.6K Jun  6 04:43 config.json
-rw-r--r-- 1 root root  158 Jun  6 04:43 feature_extractor_config.json
-rw-r--r-- 1 root root  159 Jun  6 04:43 preprocessor_config.json
-rw-r--r-- 1 root root 361M Jun  6 04:43 pytorch_model.bin


In [8]:
!ls /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints

README.md  config.yaml	generator.ckpt	renderer.ckpt  wav2vec2-base-960h


In [9]:
%%bash
set -e

cd "${REPO_DIR:-/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex}"

mkdir -p checkpoints

# Disable hf_transfer if not installed
export HF_HUB_ENABLE_HF_TRANSFER=0

python - <<'PY'
import os
from huggingface_hub import snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("Downloading IMTalker checkpoints...")

snapshot_download(
    repo_id="brianmatzelle/personaplex-7b-v1-bnb-4bit",
    repo_type="model",
    local_dir="checkpoints",
    resume_download=True,
)

print("Download complete.")

bash: line 25: warning: here-document at line 10 delimited by end-of-file (wanted `PY')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 67 files: 100%|██████████| 67/67 [00:26<00:00,  2.55it/s]


Download complete.


In [10]:
!ls /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints

README.md  model_bnb_4bit.pt  moshi


In [11]:
%%bash
set -e

cd "${REPO_DIR:-/workspace/IMTalker-PersonaPlex-Streaming-v1}"

mkdir -p checkpoints

# Disable hf_transfer if not installed
export HF_HUB_ENABLE_HF_TRANSFER=0

python - <<'PY'
import os
from huggingface_hub import snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("Downloading PersonaPlex frontend adapter checkpoints...")

snapshot_download(
    repo_id="niloy629/hdtf_preprocess",
    repo_type="dataset",
    local_dir="checkpoints",
    allow_patterns=[
        "personaplex_helium_w2v_frontend_adapter/checkpoints/*"
    ],
    resume_download=True,
)

print("Download complete.")
PY

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:08<00:00,  2.09s/it]


Download complete.


In [12]:
!ls /workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt

/workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt


In [5]:
import pathlib, os
from huggingface_hub import hf_hub_download

# ---------------------------------------------------------------------------
# Canonical paths (from run_personaplex_imtalker_source5_8998.sh)
# ---------------------------------------------------------------------------
GENERATOR_CKPT  = '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/generator.ckpt'
RENDERER_CKPT   = '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/renderer.ckpt'
MOSHI_WEIGHT    = '/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/model_bnb_4bit.pt'
ADAPTER_CKPT    = ('/workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt')
WAV2VEC_DIR     = '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/wav2vec2-base-960h'
REF_IMAGE       = '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/assets/r.png'
VOICE_PROMPT    = 'NATM0.pt'     # filename; fetched from HF voices.tgz
HF_REPO         = 'nvidia/personaplex-7b-v1'

# Loaders.py constants (from personaplex/moshi/moshi/models/loaders.py)
MIMI_NAME            = 'tokenizer-e351c8d8-checkpoint125.safetensors'
TEXT_TOKENIZER_NAME  = 'tokenizer_spm_32k_3.model'

# ---------------------------------------------------------------------------
# Validate local checkpoints (must already exist — not downloadable here)
# ---------------------------------------------------------------------------
LOCAL_REQUIRED = {
    'generator.ckpt':      GENERATOR_CKPT,
    'renderer.ckpt':       RENDERER_CKPT,
    'model_bnb_4bit.pt':   MOSHI_WEIGHT,
    'adapter checkpoint':  ADAPTER_CKPT,
    'wav2vec2-base-960h/': WAV2VEC_DIR,
    'ref image':           REF_IMAGE,
}

all_ok = True
print('=== Local checkpoint validation ===')
for label, path in LOCAL_REQUIRED.items():
    p = pathlib.Path(path)
    exists = p.exists()
    size = ''
    if exists and p.is_file():
        mb = p.stat().st_size / 1024**2
        size = f'  ({mb:.0f} MB)'
    elif exists and p.is_dir():
        n = len(list(p.iterdir()))
        size = f'  ({n} files)'
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {label}: {path}{size}')
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        'One or more local checkpoints are missing.\n'
        'These must be placed manually — they are not downloadable from public HuggingFace.\n'
        'Contact the model owner or use the file transfer instructions in PERSONAPLEX_IMTALKER_LIVE.md.'
    )

# ---------------------------------------------------------------------------
# Download Mimi tokenizer + text tokenizer from HF if not cached
# (loaders.py: get_mimi() and get_moshi_lm() both need these)
# ---------------------------------------------------------------------------
print(f'\n=== HuggingFace downloads from {HF_REPO} ===')

for hf_name, label in [
    (MIMI_NAME, 'Mimi tokenizer safetensors'),
    (TEXT_TOKENIZER_NAME, 'SentencePiece tokenizer'),
]:
    try:
        local_path = hf_hub_download(HF_REPO, hf_name)
        mb = pathlib.Path(local_path).stat().st_size / 1024**2
        print(f'  [OK] {label}: {local_path}  ({mb:.0f} MB)')
    except Exception as e:
        print(f'  [WARN] {label} download failed: {e}')

# ---------------------------------------------------------------------------
# Download + extract voices.tgz for NATM0.pt voice prompt
# (liveTry.py: _resolve_voice_prompt_path() does this automatically,
#  but we pre-download here for reliability)
# ---------------------------------------------------------------------------
print(f'\n=== Voice prompt: {VOICE_PROMPT} ===')
import tarfile

try:
    voices_tgz = pathlib.Path(hf_hub_download(HF_REPO, 'voices.tgz'))
    voices_dir = voices_tgz.parent / 'voices'
    if not voices_dir.exists():
        print(f'  Extracting {voices_tgz} ...')
        with tarfile.open(voices_tgz, 'r:gz') as tar:
            tar.extractall(path=voices_tgz.parent)
    target = voices_dir / VOICE_PROMPT
    if target.exists():
        print(f'  [OK] {target}  ({target.stat().st_size/1024:.0f} KB)')
        VOICE_PROMPT_PATH = str(target)
    else:
        print(f'  [WARN] {VOICE_PROMPT} not found in extracted voices dir {voices_dir}')
        VOICE_PROMPT_PATH = VOICE_PROMPT  # let liveTry._resolve_voice_prompt_path handle it
except Exception as e:
    print(f'  [WARN] voices.tgz download/extraction failed: {e}')
    VOICE_PROMPT_PATH = VOICE_PROMPT

print('\nCheckpoint validation complete.')

=== Local checkpoint validation ===
  [OK] generator.ckpt: /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/generator.ckpt  (593 MB)
  [OK] renderer.ckpt: /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/renderer.ckpt  (2023 MB)
  [OK] model_bnb_4bit.pt: /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/model_bnb_4bit.pt  (6659 MB)
  [OK] adapter checkpoint: /workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt  (577 MB)
  [OK] wav2vec2-base-960h/: /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/wav2vec2-base-960h  (4 files)
  [OK] ref image: /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/assets/r.png  (1 MB)

=== HuggingFace downloads from nvidia/personaplex-7b-v1 ===


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

  [OK] Mimi tokenizer safetensors: /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/tokenizer-e351c8d8-checkpoint125.safetensors  (367 MB)


tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

  [OK] SentencePiece tokenizer: /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/tokenizer_spm_32k_3.model  (1 MB)

=== Voice prompt: NATM0.pt ===


voices.tgz:   0%|          | 0.00/6.10M [00:00<?, ?B/s]

  Extracting /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/voices.tgz ...
  [OK] /workspace/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/voices/NATM0.pt  (402 KB)

Checkpoint validation complete.


## Cell 7 — Build Argument Namespace

Constructs the `argparse.Namespace` that matches exactly the arguments in
`run_personaplex_imtalker_source5_8998.sh`, using defaults from
`generator/options/base_options.py` and `LiveHeliumFMOptions.initialize()`.

In [6]:
import argparse, sys

# Simulate parsing the run-script CLI flags as an argparse.Namespace.
# Every value below corresponds 1-to-1 with a flag in the run script or
# a BaseOptions / LiveHeliumFMOptions default.

args = argparse.Namespace(
    # --- Server ---
    host='0.0.0.0',
    port=8998,
    html_path='/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/static/index_v3_binary_fullscreen.html',

    # --- IMTalker model paths (from run script) ---
    generator_path='/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/generator.ckpt',
    renderer_path='/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/renderer.ckpt',
    ref_path='/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/assets/r.png',

    # --- Helium→Wav2Vec2 adapter (from run script) ---
    adapter_path=('/workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt'),
    adapter_num_layers=6,        # matches phase2_best checkpoint
    adapter_dropout=0.1,
    stats_path='',               # unused; accepted for compatibility

    # --- Wav2Vec2 (from run script) ---
    wav2vec_model_path='/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/wav2vec2-base-960h',
    wav2vec_sec=0.96,            # training window: 0.96s × 25fps = 24 frames

    # --- PersonaPlex / Moshi (from run script) ---
    moshi_root='/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex',
    mimi_hf_repo='nvidia/personaplex-7b-v1',
    moshi_weight='/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/model_bnb_4bit.pt',
    mimi_weight='',              # empty → loaded from HF repo
    tokenizer='',                # empty → loaded from HF repo
    quantize_4bit=True,          # PersonaPlex bnb-4bit
    num_codebooks=8,
    moshi_context=0,
    voice_prompt='NATM0.pt',
    voice_prompt_dir='',         # resolved by _resolve_voice_prompt_path()
    text_prompt=(
        'You work for North South University which is a university and your name is '
        'Nabeel Mohammed. Information: you are answering Computer science related '
        'questions explicitly about models and telling about how moshi and personaplex '
        'are trained to ordinary people. So in lighter terms.'
    ),
    moshi_reply_device='cuda',
    enable_moshi_reply=True,     # mic → PersonaPlex reply → Helium → FM → avatar
    direct_reply_hidden=True,    # use Moshi layer[-2] hidden directly (no re-encode)
    moshi_cfg_coef=1.0,

    # --- FM / audio chunk settings (from run script) ---
    audio_chunk_sec=0.96,
    fm_chunk_frames=24,          # 0.96s × 25fps = 24 frames
    reply_hidden_steps_per_chunk=0,  # 0 → derived: round(24 × 12.5 / 25) = 12
    prebuffer_chunks=0,
    frame_q_backpressure=160,
    audio_path='',               # no file-drive mode; live mic only

    # --- Pose (run script passes no static pose flags → data dict has no 'pose') ---
    static_pose_zero=False,
    static_pose_values=None,

    # --- Renderer ---
    render_sub_batch=8,
    jpeg_quality=58,             # from run script

    # --- Flow Matching ODE (from run script) ---
    a_cfg_scale=1.34,
    nfe=5,
    torchdiffeq_ode_method='euler',
    ode_atol=1e-5,
    ode_rtol=1e-5,

    # --- Shared noise (from run script) ---
    shared_noise=True,
    noise_seed=42,
    noise_max_frames=5000,

    # --- Precision (from run script: --fp32 --tf32) ---
    fp32=True,
    tf32=True,
    compile_renderer=False,

    # --- Session management ---
    device='cuda',
    buffer_ms=80,
    dump_motion=True,
    dump_dir='/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/live_try_dumps_personaplex_frontend_source5_cfg134',

    # --- BaseOptions defaults used by FMGenerator ---
    fps=25.0,
    sampling_rate=16000,
    input_size=256,
    input_nc=3,
    seed=42,
    fix_noise_seed=False,
    only_last_features=True,
    average_emotion=False,
    audio_marcing=2,
    attention_window=5,
    audio_dropout_prob=0.1,
    ref_dropout_prob=0.1,
    emotion_dropout_prob=0.1,
    style_dim=512,
    dim_a=512,
    dim_h=512,
    dim_e=7,
    dim_motion=32,
    dim_c=32,
    dim_w=32,
    fmt_depth=8,
    num_heads=8,
    mlp_ratio=4.0,
    no_learned_pe=False,
    num_prev_frames=10,
    max_grad_norm=1.0,
    swin_res_threshold=128,
    window_size=8,
    # audio_adapter_mode controls AudioBridge path in FMGenerator
    # 'none' means audio_projection is a linear from audio_input_dim=768 -> dim_c=32
    audio_adapter_mode='none',
    audio_feat_dim=768,
    audio_adapter_dim=512,
    # Used by FM.sample() to distinguish Helium-driven (direct_reply_hidden) path
    rank='cuda',
    pretrained_dir='./checkpoints',
    # file_chunk_lookahead: unused (no audio_path)
    file_chunk_lookahead=0,
)

print('Argument namespace built:')
for k, v in sorted(vars(args).items()):
    print(f'  {k:35s} = {v!r}')

Argument namespace built:
  a_cfg_scale                         = 1.34
  adapter_dropout                     = 0.1
  adapter_num_layers                  = 6
  adapter_path                        = '/workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt'
  attention_window                    = 5
  audio_adapter_dim                   = 512
  audio_adapter_mode                  = 'none'
  audio_chunk_sec                     = 0.96
  audio_dropout_prob                  = 0.1
  audio_feat_dim                      = 768
  audio_marcing                       = 2
  audio_path                          = ''
  average_emotion                     = False
  buffer_ms                           = 80
  compile_renderer                    = False
  device                              = 'cuda'
  dim_a                               = 512
  dim_c                               = 32
  dim_e                               

## Cell 8 — In-Process Server (Option A: Integrated)

Builds the FastAPI app directly from `liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.build_app()`
and launches it in a background thread via uvicorn. This is the same entrypoint as the run script —
no code duplication, no reimplementation.

**Skip this cell and use Cell 9 (subprocess) if you prefer isolation.**

In [8]:
!pip install einops

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moshi-personaplex 0.1.0 requires sounddevice==0.5, which is not installed.
moshi-personaplex 0.1.0 requires sphn<0.2,>=0.1.4, which is not installed.
moshi-personaplex 0.1.0 requires aiohttp<3.11,>=3.10.5, but you have aiohttp 3.14.0 which is incompatible.
moshi-personaplex 0.1.0 requires einops==0.7, but you have einops 0.8.2 which is incompatible.
moshi-personaplex 0.1.0 requires huggingface-hub<0.25,>=0.24, but you have huggingface-hub 0.36.2 which is incompatible.
moshi-personaplex 0.1.0 requires safetensors<0.5,>=0.4.0, but you have safetensors 0.7.0 which is incompatible.
moshi-personaplex 0.1.0 requires sentencepiece==0.2, but you have sentencepiece 0.2.1 which is incompatible.
moshi-personaplex 0.1.0 requires torch<2.5,>=2.2.0, but you have torch 2.8.0+cu128 which is incompatible.


In [10]:
!pip install sphn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 27.5 MB/s  0:00:00 eta 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moshi-personaplex 0.1.0 requires sounddevice==0.5, which is not installed.
moshi-personaplex 0.1.0 requires aiohttp<3.11,>=3.10.5, but you have aiohttp 3.14.0 which is incompatible.
moshi-personaplex 0.1.0 requires einops==0.7, but you have einops 0.8.2 which is incompatible.
moshi-personaplex 0.1.0 requires huggingface-hub<0.25,>=0.24, but you have huggingface-hub 0.36.2 which is incompatible.
moshi-personaplex 0.1.0 requires safetensors<0.5,>=0.4.0, but you have safetensors 0.7.0 which is incompatible.
moshi-personaplex 0.1.0 requires sentencepiece==0.2, but you have sentencepiece 0.2.1 which is incompatible.
moshi-personaplex 0.1.0 requires sphn<0.2,>=0.1.4, but you have sphn 0.2.1 which is incompatible.
moshi-personaplex 0.

In [14]:
import sys, threading, uvicorn, logging, importlib.util, pathlib, inspect

# The PersonaPlex moshi fork is at personaplex/checkpoints/moshi, NOT personaplex/moshi.
# Confirmed via: grep -r "quantize_4bit" /workspace --include="loaders.py" -l
#   /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/moshi/moshi/models/loaders.py
MOSHI_FORK_ROOT = '/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/moshi'

# ── 1. Put the fork root first, then flush ALL cached moshi modules ──────────
# Must happen BEFORE exec_module so liveTry.py's module-level
# `from moshi.models import loaders` resolves to the fork, not the standard moshi.
if MOSHI_FORK_ROOT in sys.path:
    sys.path.remove(MOSHI_FORK_ROOT)
sys.path.insert(0, MOSHI_FORK_ROOT)
print(f'sys.path[0] = {MOSHI_FORK_ROOT}')

_stale = [k for k in list(sys.modules.keys()) if k == 'moshi' or k.startswith('moshi.')]
for _k in _stale:
    del sys.modules[_k]
print(f'Flushed {len(_stale)} stale moshi modules')

# ── 2. Verify the fork has quantize_4bit ─────────────────────────────────────
import moshi.models.loaders as _ml
print(f'loaders.__file__: {_ml.__file__}')
_params = list(inspect.signature(_ml.get_moshi_lm).parameters.keys())
print(f'get_moshi_lm params: {_params}')
assert 'quantize_4bit' in _params, (
    f'Fork at {MOSHI_FORK_ROOT} still lacks quantize_4bit.\n'
    f'Loaded from: {_ml.__file__}'
)
print('[OK] PersonaPlex fork confirmed — proceeding to load entrypoint')

# ── 3. Load the entrypoint ────────────────────────────────────────────────────
# sys.modules['moshi.models.loaders'] is now the fork version.
# liveTry.py's module-level `from moshi.models import loaders` will pick it from cache.
spec = importlib.util.spec_from_file_location(
    'imtalker_live',
    '/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/personaplex_imtalker_streaming_server.py',
)
imtalker_live = importlib.util.module_from_spec(spec)
spec.loader.exec_module(imtalker_live)

print('[notebook] Building FastAPI app...')
app = imtalker_live.build_app(args)
print('[notebook] App built. Starting uvicorn on 0.0.0.0:8998 ...')

class _UvicornThread(threading.Thread):
    def __init__(self, app, host, port):
        super().__init__(daemon=True, name='uvicorn-server')
        config = uvicorn.Config(
            app, host=host, port=port, log_level='info',
            ws_ping_interval=20, ws_ping_timeout=30,
        )
        self.server = uvicorn.Server(config)
    def run(self):
        self.server.run()
    def stop(self):
        self.server.should_exit = True

_server_thread = _UvicornThread(app, args.host, args.port)
_server_thread.start()

import time, httpx
for attempt in range(30):
    time.sleep(2)
    try:
        r = httpx.get(f'http://127.0.0.1:{args.port}/health', timeout=3)
        if r.status_code == 200:
            print(f'[notebook] Health check OK: {r.json()}')
            break
    except Exception:
        print(f'[notebook] Waiting for server... attempt {attempt+1}/30')
else:
    raise RuntimeError('Server did not become healthy after 60s')


sys.path[0] = /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/moshi
Flushed 20 stale moshi modules
loaders.__file__: /workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/moshi/moshi/models/loaders.py
get_moshi_lm params: ['filename', 'copy_missing_weights', 'device', 'dtype', 'delays', 'cpu_offload', 'quantize_4bit', 'num_codebooks', 'context']
[OK] PersonaPlex fork confirmed — proceeding to load entrypoint
[notebook] Building FastAPI app...


Some weights of Wav2VecModel were not initialized from the model checkpoint at /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[Model Stats] Parameters: 133,193,248 | Trainable: 38,821,536
[liveTryHeliumFM][FM] loaded in 868ms missing=211 unexpected=0
[liveTryHeliumFM][renderer] loaded in 2818ms missing=0 unexpected=139


Some weights of Wav2VecModel were not initialized from the model checkpoint at /workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[liveTryHeliumFrontendFM][adapter] frontend-fp32 loaded in 647ms path=/workspace/IMTalker-PersonaPlex-Streaming-v1/checkpoints/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt layers=6
[liveTryHeliumStudioFM][wav2vec] loaded in 1ms path=/workspace/IMTalker-PersonaPlex-Streaming-v1/IMTalker/checkpoints/wav2vec2-base-960h
[liveTryHeliumStudioFM] using direct Moshi reply hidden; batch Helium extractor skipped
[liveTryHeliumFM] shared noise buf: (1, 5000, 32)
[liveTryHeliumStudioFM][warmup] raw_helium=skipped direct_hidden raw_steps=12
[liveTryHeliumFM][warmup] fm=40ms motion=(24, 32)
[liveTryHeliumFM][warmup] renderer=134ms
[liveTryHeliumFM][warmup] jpeg=1ms
[liveTryHeliumFM] ready — total startup 4676ms fm_chunk=24 render_sub=8 dtype=torch.float32
[liveTryHeliumFM_ws_binary] eager-loading Moshi/PersonaPlex reply engine
[liveTry] loading Moshi repo=nvidia/personaplex-7b-v1 root=/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex device=cuda cfg=1.

INFO:     Started server process [1137]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8998 (Press CTRL+C to quit)


[notebook] App built. Starting uvicorn on 0.0.0.0:8998 ...
INFO:     127.0.0.1:46788 - "GET /health HTTP/1.1" 200 OK
[notebook] Health check OK: {'ok': True, 'stage': 'live_moshi_reply_helium_fm', 'uptime_sec': 38.837, 'loaded': True}


In [13]:
!grep -r "quantize_4bit" /workspace --include="loaders.py" -l

/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/moshi/build/lib/moshi/models/loaders.py
/workspace/IMTalker-PersonaPlex-Streaming-v1/personaplex/checkpoints/moshi/moshi/models/loaders.py


## Cell 9 — Subprocess Server (Option B: Isolated)

Launches the run script as a managed subprocess exactly as `run_personaplex_imtalker_source5_8998.sh` does.
**Use this instead of Cell 8 if you want full process isolation.**
Only run ONE of Cell 8 or Cell 9.

In [ ]:
# ── OPTION B: subprocess launch ──────────────────────────────────────────────
# Uncomment and run this cell INSTEAD of Cell 8 if you prefer process isolation.
# ─────────────────────────────────────────────────────────────────────────────

# import subprocess, os, sys, time, pathlib, httpx, threading
#
# LOG_PATH = '/workspace/IMTalker/logs/live_personaplex_imtalker_source5_8998.log'
# pathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
#
# env = os.environ.copy()
# env['PYTHONPATH']           = '/workspace/IMTalker:/workspace/personaplex_bnb4/moshi'
# env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# env['CUDA_VISIBLE_DEVICES'] = '0'
# # HF_TOKEN must already be in env from Cell 5
#
# cmd = [
#     sys.executable, '-u',
#     '/workspace/IMTalker/personaplex_imtalker_streaming_server.py',
#     '--generator_path', '/workspace/IMTalker/checkpoints/generator.ckpt',
#     '--renderer_path',  '/workspace/IMTalker/checkpoints/renderer.ckpt',
#     '--adapter_path',   ('/workspace/exps/personaplex_frontend_adapter'
#                          '/personaplex_helium_w2v_frontend_adapter/checkpoints'
#                          '/phase2_best_wav2vec_final_loss.pt'),
#     '--adapter_num_layers', '6',
#     '--adapter_dropout',    '0.1',
#     '--wav2vec_model_path', '/workspace/IMTalker/checkpoints/wav2vec2-base-960h',
#     '--ref_path',           '/workspace/IMTalker/assets/2_vid_robert.png',
#     '--host', '0.0.0.0',
#     '--port', '8998',
#     '--device', 'cuda',
#     '--enable_moshi_reply',
#     '--direct_reply_hidden',
#     '--moshi_root',          '/workspace/personaplex_bnb4',
#     '--mimi_hf_repo',        'nvidia/personaplex-7b-v1',
#     '--moshi_weight',        '/workspace/personaplex_bnb4/model_bnb_4bit.pt',
#     '--quantize_4bit',
#     '--num_codebooks', '8',
#     '--moshi_reply_device', 'cuda',
#     '--moshi_cfg_coef', '1.0',
#     '--voice_prompt', 'NATM0.pt',
#     '--text_prompt', (
#         'You work for North South University which is a university and your name is '
#         'Nabeel Mohammed. Information: you are answering Computer science related '
#         'questions explicitly about models and telling about how moshi and personaplex '
#         'are trained to ordinary people. So in lighter terms.'
#     ),
#     '--a_cfg_scale', '1.34',
#     '--nfe', '5',
#     '--wav2vec_sec', '0.96',
#     '--audio_chunk_sec', '0.96',
#     '--fm_chunk_frames', '24',
#     '--reply_hidden_steps_per_chunk', '0',
#     '--prebuffer_chunks', '0',
#     '--frame_q_backpressure', '160',
#     '--render_sub_batch', '8',
#     '--jpeg_quality', '58',
#     '--dump_motion',
#     '--dump_dir', '/workspace/IMTalker/live_try_dumps_personaplex_frontend_source5_cfg134',
#     '--shared_noise',
#     '--noise_seed', '42',
#     '--noise_max_frames', '5000',
#     '--fp32',
#     '--tf32',
# ]
#
# log_file = open(LOG_PATH, 'w', buffering=1)
#
# _proc = subprocess.Popen(
#     cmd,
#     stdout=log_file,
#     stderr=subprocess.STDOUT,
#     env=env,
#     cwd='/workspace/IMTalker',
# )
# print(f'[notebook] Server PID={_proc.pid}, log={LOG_PATH}')
#
# for attempt in range(60):
#     time.sleep(3)
#     if _proc.poll() is not None:
#         raise RuntimeError(f'Server exited early (code={_proc.returncode}). Check {LOG_PATH}')
#     try:
#         r = httpx.get('http://127.0.0.1:8998/health', timeout=3)
#         if r.status_code == 200:
#             print(f'[notebook] Health OK: {r.json()}')
#             break
#     except Exception:
#         print(f'[notebook] Waiting... attempt {attempt+1}/60')
# else:
#     raise RuntimeError('Server did not become healthy after 180s')

print('Cell 9 (subprocess mode) is commented out. Running Cell 8 (in-process) instead.')
print('Uncomment Cell 9 if you prefer subprocess isolation.')

## Cell 10 — Healthy Startup Verification

From `PERSONAPLEX_IMTALKER_LIVE.md`, healthy startup includes:
```
frontend-fp32 loaded
using direct Moshi reply hidden
voice prompt: .../NATM0.pt
installed PersonaPlex graphed hidden capture
serving /workspace/IMTalker/static/index_v3_binary_fullscreen.html
Uvicorn running on http://0.0.0.0:8998
```

In [ ]:
import httpx, time

PORT = 8998

# 1. Health endpoint (FastAPI /health route from liveTryHeliumFrontendDeque...)
print('=== Health check ===')
r = httpx.get(f'http://127.0.0.1:{PORT}/health', timeout=10)
health = r.json()
print(f'  status_code: {r.status_code}')
for k, v in health.items():
    print(f'  {k}: {v}')

assert health.get('ok') is True, f'Health endpoint returned ok=False: {health}'

# 2. Index HTML (served by FileResponse from /workspace/IMTalker/static/index_v3_binary_fullscreen.html)
print('\n=== HTML index ===')
r2 = httpx.get(f'http://127.0.0.1:{PORT}/', timeout=10)
html_size = len(r2.content)
print(f'  status_code: {r2.status_code}')
print(f'  content-type: {r2.headers.get("content-type", "")}')
print(f'  size: {html_size} bytes')
# PERSONAPLEX_IMTALKER_LIVE.md expects: 200 35708
if html_size < 10000:
    print(f'  WARNING: HTML smaller than expected ({html_size} < 10000)')
assert r2.status_code == 200, f'Index returned {r2.status_code}'

print('\nServer is healthy and serving the IMTalker frontend.')

## Cell 11 — Access URL

Displays the public URL for the IMTalker frontend. On RunPod, port 8998 is
exposed via the pod's HTTP service URL.

In [ ]:
import os

PORT = 8998

# RunPod exposes HTTP ports via a predictable URL pattern.
# The pod ID is available in RUNPOD_POD_ID environment variable.
pod_id = os.environ.get('RUNPOD_POD_ID', '')
if pod_id:
    public_url = f'https://{pod_id}-{PORT}.proxy.runpod.net'
    print(f'Public URL (RunPod):  {public_url}')
else:
    print('RUNPOD_POD_ID not set — running locally or on a non-RunPod host.')
    public_url = f'http://localhost:{PORT}'

print(f'Local URL:            http://127.0.0.1:{PORT}/')
print(f'WebSocket endpoint:   ws://127.0.0.1:{PORT}/ws/conversation')
print()
print('Open the URL above in a Chromium-based browser.')
print('Click the microphone button to begin the live conversation.')
print()
print('The pipeline (from liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py):')
print('  Browser mic (48 kHz) → MoshiOnlyEngineWithHidden → HeliumToWav2VecFrontendAdapter')
print('  → FMGenerator.sample() → IMTRenderer.decode() → binary WS → browser')

## Cell 12 — Real-Time Monitoring

Polls the `/health` endpoint and streams GPU stats. Run for as long as you need to observe the pipeline.
Interrupt the cell to stop monitoring.

In [ ]:
import time, httpx, subprocess, sys

PORT = 8998
POLL_INTERVAL_S = 10

def _gpu_stats():
    """Returns a one-line GPU VRAM/utilization string."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi',
             '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
             '--format=csv,noheader,nounits'],
            text=True, timeout=5
        ).strip()
        parts = [p.strip() for p in out.split(',')]
        name, mem_used, mem_total, util, temp = parts
        return (f'{name} | VRAM {mem_used}/{mem_total} MiB '
                f'| GPU util {util}% | {temp}°C')
    except Exception as e:
        return f'nvidia-smi error: {e}'

print(f'Monitoring health every {POLL_INTERVAL_S}s. Interrupt to stop.\n')
try:
    iteration = 0
    while True:
        iteration += 1
        ts = time.strftime('%H:%M:%S')
        try:
            r = httpx.get(f'http://127.0.0.1:{PORT}/health', timeout=5)
            h = r.json()
            uptime = h.get('uptime_sec', 0)
            stage  = h.get('stage', '?')
            loaded = h.get('loaded', '?')
            health_str = f'OK  stage={stage} loaded={loaded} uptime={uptime:.0f}s'
        except Exception as e:
            health_str = f'ERROR: {e}'

        gpu_str = _gpu_stats()
        print(f'[{ts}] #{iteration:04d}  health={health_str}')
        print(f'           gpu={gpu_str}')
        time.sleep(POLL_INTERVAL_S)
except KeyboardInterrupt:
    print('\nMonitoring stopped.')

## Cell 13 — Graceful Shutdown

In [ ]:
# --- Option A shutdown (in-process uvicorn thread) ---
try:
    _server_thread.stop()
    _server_thread.join(timeout=10.0)
    print('[notebook] In-process server stopped.')
except NameError:
    pass  # Cell 8 was not run

# --- Option B shutdown (subprocess) ---
try:
    if _proc.poll() is None:
        _proc.terminate()
        _proc.wait(timeout=15)
        print(f'[notebook] Subprocess PID={_proc.pid} terminated.')
    else:
        print(f'[notebook] Subprocess already exited (code={_proc.returncode}).')
except NameError:
    pass  # Cell 9 was not run

# Release GPU memory
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
print('[notebook] GPU memory released.')

## Appendix A — Architecture Summary

### Model loading sequence (derived from source)

```
LiveHeliumFMEngine.__init__(args)
  ├─ _load_fm(args, device)
  │     FMGenerator(args)  [generator/FM.py]
  │       AudioEncoder (Wav2Vec2, frozen)
  │       FlowMatchingTransformer [generator/FMT.py]
  │       audio_projection: Linear(768→32) + LN + SiLU
  │     torch.load(generator.ckpt)
  │     fm.load_state_dict(cleaned, strict=False)
  ├─ _load_renderer(args, device, dtype)
  │     IMTRenderer(args)  [renderer/models.py]
  │       IdentityEncoder (dense_feature_encoder)
  │       MotionEncoder   (latent_token_encoder)
  │       MotionDecoder   (latent_token_decoder)
  │       SynthesisNetwork (frame_decoder)
  │       IdentidyAdaptive (adapt)
  │       CrossAttention × 6 (imt)
  │     torch.load(renderer.ckpt)
  │     renderer.load_state_dict(cleaned, strict=False)
  ├─ StudioNativeLiveAdapter.__init__(wav2vec_model_path, num_layers=6, dropout=0.1)
  │     HeliumToWav2VecFrontendAdapter(num_layers=6)  [generator/helium_w2v_frontend_adapter.py]
  │       input_norm LN(4096) → input_proj Linear(4096→768) → pos_conv Conv1d(768,768,128) → 6×TransformerEncoderLayer → final_norm
  │     Wav2VecModel.from_pretrained(wav2vec_model_path, local_files_only=True)  [generator/wav2vec2.py]
  │       Frozen Wav2Vec2-base-960h
  │     torch.load(adapter_path)  →  adapter.load_state_dict(state, strict=True)
  ├─ Pre-compute ref image features
  │     dense_feature_encoder(ref_tensor) → (f_r, g_r)
  │     latent_token_encoder(ref_tensor) → ref_x
  │     adapt(ref_x, g_r) → latent_token_decoder → m_r
  └─ MoshiOnlyEngineWithHidden.__init__()   (eager-loaded by enable_moshi_reply=True)
        loaders.get_mimi(mimi_weight, device)  [personaplex/moshi/moshi/models/loaders.py]
        loaders.get_moshi_lm(moshi_weight, dtype=bfloat16, quantize_4bit=True)
        LMGen(lm, cfg_coef=1.0, condition_tensors=cond_tensors)
        mimi.streaming_forever(1)
        lm_gen.streaming_forever(1)
        _apply_system_prompts()   → load_voice_prompt_embeddings(NATM0.pt)
        _install_graph_hidden_capture()   → patches lm_gen._step to return layer_hidden
```

### Per-step inference (80ms chunks @ 12.5 Hz)

```
MoshiOnlyEngineWithHidden._step(pcm24)
  mimi.encode(chunk)               → codes [1, 8, 1]
  lm_gen._step(codes[:, :, :1])    → (tokens, transformer_out, layer_hidden)
  layer_hidden[:1, -1:]            → helium_hidden [1, 1, 4096]  @ 12.5 Hz
  mimi.decode(tokens[:, 1:])       → reply_pcm [1920 samples @ 24kHz]

Every 12 hidden steps (≈0.96s = fm_chunk_frames=24 @ 25fps):
  helium_chunk = cat(12 × [1, 4096])           → [12, 4096]
  StudioNativeLiveAdapter.forward_single(helium_deque[100], target_len=200)
    HeliumToWav2VecFrontendAdapter(helium_deque, target_len=200) → frontend50 [1, 200, 768]
    Wav2Vec2.encode_from_projected_frontend(frontend50)          → final50   [1, 200, 768]
    interpolate(final50, size=24)                                → feat_25   [24, 768]
  FMGenerator.sample(data={a_feat: feat_25, ref_x}, a_cfg_scale=1.34, nfe=5)
    odeint(sample_chunk, x0[B,24,32], linspace(0,1,5))
    → motion [1, 24, 32]
  IMTRenderer:
    adapt(motion, g_r) → latent_token_decoder → m_c (motion maps)
    decode(m_c, m_r, f_r) → frames_rgb [24, 512, 512, 3]
  JPEG encode × 24 frames (ThreadPoolExecutor, 4 workers)
  ws_av_binary_codec.pack_av_frame() × 24 → binary WS → browser
```

### Dependency Report

| Package | Version | Source |
|---|---|---|
| numpy | <2.0.0 | requirement.txt |
| opencv-python | <4.10.0 | requirement.txt |
| transformers | ==4.30.2 | requirement.txt |
| torchdiffeq | ==0.2.5 | requirement.txt; `generator/FM.py` `from torchdiffeq import odeint` |
| timm | ==1.0.9 | requirement.txt |
| fastapi | ==0.115.6 | requirement.txt; server entrypoint |
| pydantic | ==2.10.4 | requirement.txt |
| av | ==12.0.0 | requirement.txt |
| face_alignment | ==1.4.1 | requirement.txt |
| uvicorn | [standard] | `liveTry.py` main() |
| sentencepiece | latest | `liveTry.py` SentencePieceProcessor |
| huggingface-hub | >=0.20 | `liveTry.py` hf_hub_download |
| safetensors | >=0.4.0 | `personaplex/moshi/moshi/models/loaders.py` |
| bitsandbytes | >=0.43.0 | `loaders.get_moshi_lm(quantize_4bit=True)` |
| accelerate | >=0.26.0 | bitsandbytes dependency |

### RunPod Requirements

| Requirement | Value |
|---|---|
| GPU VRAM | ≥24 GB (bnb-4bit PersonaPlex 7B) |
| System RAM | ≥32 GB |
| CUDA | ≥12.1 |
| Port | 8998 (HTTP + WebSocket) |
| HF_TOKEN | Required for `nvidia/personaplex-7b-v1` |
| FFmpeg | System package |

### Known Issues / Code Notes

1. **`get_moshi_lm` `quantize_4bit` parameter** — The standard `kyutai/moshi` loaders do not have this flag. The PersonaPlex fork at `/workspace/personaplex_bnb4/moshi` extends loaders to add it. If `PYTHONPATH` does not put `personaplex_bnb4/moshi` first, `quantize_4bit` will fail.

2. **`_install_graph_hidden_capture` dual path** — The code tries the `prepare_step_input` / `process_transformer_output` API (newer PersonaPlex LMGen), then falls back to monkey-patching `forward_text` on the LM model. Both paths return `(tokens, transformer_out, layer_hidden)` tuples.

3. **`audio_adapter_mode`** — `FMGenerator` in the run configuration uses `mode='none'`, meaning `audio_adapter = nn.Identity()` and `audio_projection = Linear(768→32)+LN+SiLU`. The Wav2Vec2 768-dim features are projected directly to `dim_c=32`.

4. **Reference image** — The run script uses `/workspace/IMTalker/assets/2_vid_robert.png` which is not in the local repo. The repo has `source_1.png` through `source_11.png`. If `2_vid_robert.png` is missing, substitute any 512×512 face image (the renderer resizes to 512×512 via `load_ref_image()`).

### Appendix B -- FlashTalk-v3 Architecture Migration

`personaplex_imtalker_streaming_server.py` reproduces the pipeline documented above with one structural change: `MoshiOnlyEngineWithHidden`'s per-step loop and the FM/renderer chunk loop no longer run in a single "gpu-producer" thread. Instead:

```
[PersonaPlexThread] PersonaPlexEngine.run_streaming(mic_queue, helium_queue, stop_event)
    Mimi encode -> lm_gen._step (graphed hidden capture) -> Mimi decode
    pushes (step_id, helium_hidden[1,4096], reply_pcm[1920], ts, codes, is_speech)
  -> helium_queue
[IMTalkerThread] PersonaPlexConversationSession._imtalker_loop
    accumulates hidden_steps_per_chunk steps -> HeliumTokenDeque.push_batch
    IMTalkerFrontendAdapter -> FMGenerator.sample -> IMTRenderer -> JPEG encode
    pushes ChunkBundle -> dispatch_queue
[asyncio] PersonaPlexConversationSession._dispatcher
    packs ws_av_binary_codec.pack_av_frame() and sends binary AV01 frames, paced @fps
```

Same checkpoints, same `HeliumToWav2VecFrontendAdapter`/`FMGenerator`/`IMTRenderer` math, same binary wire protocol and `server_ready` handshake fields -- only the threading/queue/session structure changed, removing the render-stalls-generation bottleneck of the single-thread design.
